In [1]:
import pandas as pd
import numpy as np

## Read in the kinase result from the previous step

Filtering criteria

- 1. All the structure must have resolution > 4 Angstroms and we only take X-ray or Cryo-EM
- 2. There shouldn't be any missing residues/atoms that would cause the key labels to have "none"
- 3. Eliminate chains that are not the key kinase domain
- 4. Separate out mutants from WT
- 5. Separate phosphorylated protein from nonphosphorylated ones for WT

In [2]:
# filtering on 1 and 2
df = pd.read_csv("../step_2_pdb_extraction/step_2_pdb_kinase_metadata.csv")
cols = ["spatial_label","dihedral_label","chelix_label", "saltbr_label", "actloop_nt_label", "actloop_ct_label"]
mask = ~df[cols].apply(lambda col: col.str.contains("none", na=False)).any(axis=1)
df = df[mask]
df = df.dropna(subset=cols, how='any')

df_highres = df[(df['resolution'] < 4.5) & (df['resolution'] > 0.0)]

In [3]:
# calculate canonical uniprot IDs and assign cannonica gene names
import ast

df_highres["canonical_uniprot"] = df_highres["uniprot_id_pdb_api"].apply(lambda s: ast.literal_eval(s)[0] if pd.notna(s) else None)

mapping_df = pd.read_csv("../step_1_uniprot_extraction/dunbrack_active_kinase_mapping.txt", delimiter=" ")
uniprot_to_gene = mapping_df.set_index("Uniprot")["SwissProt_ID"].to_dict()
df_highres["canonical_gene_name"] = df_highres["canonical_uniprot"].map(uniprot_to_gene)

/tmp/ipykernel_1502285/3500535226.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_highres["canonical_uniprot"] = df_highres["uniprot_id_pdb_api"].apply(lambda s: ast.literal_eval(s)[0] if pd.notna(s) else None)
/tmp/ipykernel_1502285/3500535226.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_highres["canonical_gene_name"] = df_highres["canonical_uniprot"].map(uniprot_to_gene)


In [4]:
# filter 3 for only the kinase regions (excluding pseudo-kinases)
mapping_df = pd.read_csv("../step_1_uniprot_extraction/step_1_kinase_domain_info.csv")
mapping_uniprot_start = mapping_df.set_index("uniprot_id")["uniprot_start"].to_dict()
mapping_uniprot_end = mapping_df.set_index("uniprot_id")["uniprot_end"].to_dict()
df_highres["kinase_start"] = df_highres["canonical_uniprot"].map(mapping_uniprot_start)
df_highres["kinase_end"] = df_highres["canonical_uniprot"].map(mapping_uniprot_end)

def overlap_percent(row):
    uniprot_start = row['uniprot_align_start']
    uniprot_end = row['uniprot_align_end']
    kinase_start = row['kinase_start']
    kinase_end = row['kinase_end']
    overlap_start = max(uniprot_start, kinase_start)
    overlap_end = min(uniprot_end, kinase_end)
    alignment_length = uniprot_end - uniprot_start + 1

    if alignment_length < 100: return 1
    
    # If no overlap, return 0
    if overlap_end < overlap_start:
        return 0.0
    
    overlap_length = overlap_end - overlap_start + 1
    kinase_length = kinase_end - kinase_start + 1
    overlap_percentage = overlap_length / kinase_length
    
    return overlap_percentage

df_highres["overlap_percent"] = df_highres.apply(overlap_percent, axis=1)


/tmp/ipykernel_1502285/3790596208.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_highres["kinase_start"] = df_highres["canonical_uniprot"].map(mapping_uniprot_start)
/tmp/ipykernel_1502285/3790596208.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_highres["kinase_end"] = df_highres["canonical_uniprot"].map(mapping_uniprot_end)
/tmp/ipykernel_1502285/3790596208.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [5]:
df_highres = df_highres[(df_highres['overlap_percent'] >= 0)]
df_highres.shape

(6703, 73)

### Separation with steps 4 and 5

In [6]:
df_highres['covalent_info'].isna().sum() / df_highres['covalent_info'].isna().shape[0]

0.936147993435775

In [7]:
df_highres['phosphorylations'].isna().sum() / df_highres['phosphorylations'].isna().shape[0]

0.6981948381321796

In [8]:
df_highres['mutations'].isna().sum() / df_highres['mutations'].isna().shape[0]

0.7143070267044607

In [9]:
df_highres.to_csv("highres_kinase_metadata.csv", index=False)

In [10]:
df_mut = df_highres[(df_highres['num_mutations'] > 0)]

In [11]:
df_mut[['structure_id', 'chain_id', 'canonical_uniprot',
       'kinase_family', 'canonical_gene_name',
       'activity_label', 'spatial_label', 'dihedral_label', 'chelix_label',
       'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
       'ligands', 'ligand_type', 'num_ligands', 'lys_residue',
       'glu_residue', 'asp_residue', 'phe_residue', 'gly_residue',
       'dfg_residue', 'hrd_residue', 'lys_glu_distance', 'saltbridge_distance',
       'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
       'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
       'sequence_length', 'first_residue', 'phosphorylations', 'mutations', 'is_covalent_binder',
       'covalent_info', 'method', 'resolution', 'deposition_date']].to_csv("mut/human_kinases_mut.csv", index=False)

In [12]:
df_covalent = df_highres[(df_highres['is_covalent_binder'] == True)]

In [13]:
df_covalent[['structure_id', 'chain_id', 'canonical_uniprot',
       'kinase_family', 'canonical_gene_name',
       'activity_label', 'spatial_label', 'dihedral_label', 'chelix_label',
       'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
       'ligands', 'ligand_type', 'num_ligands', 'lys_residue',
       'glu_residue', 'asp_residue', 'phe_residue', 'gly_residue',
       'dfg_residue', 'hrd_residue', 'lys_glu_distance', 'saltbridge_distance',
       'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
       'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
       'sequence_length', 'first_residue', 'phosphorylations', 'mutations', 'is_covalent_binder',
       'covalent_info', 'method', 'resolution', 'deposition_date']].to_csv("cov/human_kinases_cov.csv", index=False)

In [14]:
df_wt_nocov = df_highres[(df_highres['num_mutations'] == 0) & (df_highres['is_covalent_binder'] == False)]

In [15]:
df_wt_nocov[['structure_id', 'chain_id', 'canonical_uniprot',
       'kinase_family', 'canonical_gene_name',
       'activity_label', 'spatial_label', 'dihedral_label', 'chelix_label',
       'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
       'ligands', 'ligand_type', 'num_ligands', 'lys_residue',
       'glu_residue', 'asp_residue', 'phe_residue', 'gly_residue',
       'dfg_residue', 'hrd_residue', 'lys_glu_distance', 'saltbridge_distance',
       'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
       'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
       'sequence_length', 'first_residue', 'phosphorylations', 'method', 'resolution', 'deposition_date']].to_csv("wt_nocov/human_kinases_wt_nocov.csv", index=False)

In [16]:
df_wt_nophos_nocov = df_highres[(df_highres['num_mutations'] == 0) & (df_highres['num_phosphorylations'] == 0) & (df_highres['is_covalent_binder'] == False)]

In [17]:
df_wt_nophos_nocov[['structure_id', 'chain_id', 'canonical_uniprot',
       'kinase_family', 'canonical_gene_name',
       'activity_label', 'spatial_label', 'dihedral_label', 'chelix_label',
       'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
       'ligands', 'ligand_type', 'num_ligands', 'lys_residue',
       'glu_residue', 'asp_residue', 'phe_residue', 'gly_residue',
       'dfg_residue', 'hrd_residue', 'lys_glu_distance', 'saltbridge_distance',
       'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
       'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
       'sequence_length', 'first_residue', 'method', 'resolution', 'deposition_date']].to_csv("wt_nophos_nocov/human_kinases_wt_nophos_nocov.csv", index=False)

In [18]:
df_mut.shape, df_covalent.shape, df_wt_nocov.shape, df_wt_nophos_nocov.shape

((1915, 73), (428, 73), (4564, 73), (2900, 73))

## Canonical Sequences for Each Group

Generate the filtered csv after removing trivial duplicates (keeping all apo and one representative holo per ligand)

In [19]:
pho = "_nocov"

kinase_base = pd.read_csv(f"wt{pho}/human_kinases_wt{pho}.csv")
filtered_kinase_base = kinase_base.dropna(subset = ['structure_id', 'chain_id', 'canonical_uniprot', 
                                                    'canonical_gene_name', 'kinase_family', 'activity_label',
                                    'spatial_label', 'dihedral_label', 'chelix_label',
                                    'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
                                    'asp_residue', 'phe_residue', 'gly_residue',
                                    'ligands', 'ligand_type', 'method', 'resolution', 'deposition_date'])

print(filtered_kinase_base.shape)

# uniprot_id_to_keep = list(set([k.split("_")[0] for k in canonical_sequences.keys()]))
# filtered_kinase_base = kinase_base[kinase_base['canonical_uniprot'].isin(uniprot_id_to_keep)]
filtered_kinase_base['structural_label'] = filtered_kinase_base['ligands'].apply(
    lambda x: 'apo' if x == 'No_ligand' else 'holo'
)

filtered_apo = filtered_kinase_base[filtered_kinase_base['structural_label'] == 'apo']

filtered_kinase_base['ligand_label'] = filtered_kinase_base['ligands'].apply(
    lambda x: ";".join([lig.split(":")[0] for lig in x.split(";")])
)

# further filter down based on labels, extract the higer resolution ones 

filtered_holo = (
    filtered_kinase_base[filtered_kinase_base['structural_label'] == 'holo']
    .sort_values(by='resolution')  # Sort by resolution ascending
    .drop_duplicates(
        subset=[
            'canonical_uniprot', 'kinase_family', 'activity_label', 'spatial_label',
            'dihedral_label', 'chelix_label', 'saltbr_label', 
            'actloop_nt_label', 'actloop_ct_label', 'spine_label',
            'asp_residue', 'phe_residue', 'gly_residue',
            'ligand_label', 'ligand_type', 'phosphorylations'
        ],
        keep='first'  # Keep the first (lowest resolution) after sorting
    )
)

(4451, 41)


/tmp/ipykernel_1502285/337453657.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_kinase_base['structural_label'] = filtered_kinase_base['ligands'].apply(
/tmp/ipykernel_1502285/337453657.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_kinase_base['ligand_label'] = filtered_kinase_base['ligands'].apply(


In [20]:
print(filtered_apo.shape, filtered_holo.shape)
filtered_df = pd.concat([filtered_apo, filtered_holo], ignore_index=True)
filtered_df

(327, 42) (3009, 43)


,structure_id,chain_id,canonical_uniprot,kinase_family,canonical_gene_name,activity_label,spatial_label,dihedral_label,chelix_label,saltbr_label,...,phe_chi1,phe_chi2,sequence_length,first_residue,phosphorylations,method,resolution,deposition_date,structural_label,ligand_label
0,1B6C,B,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,300.29,176.12,342,162,NaN,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
1,1B6C,D,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,300.53,176.04,342,162,NaN,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
2,1B6C,F,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,300.41,176.19,342,162,NaN,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
3,1B6C,H,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,300.35,176.04,342,162,NaN,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
4,1BLX,A,Q00534,CMGC,CDK6_HUMAN,Inactive,DFGin,BLBplus,Chelix-out,SaltBr-out,...,58.33,85.97,326,1,NaN,X-RAY DIFFRACTION,1.90,1999-06-01,apo,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3331,3OMV,B,P04049,TKL,RAF1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-in,...,285.33,59.46,307,312,NaN,X-RAY DIFFRACTION,4.00,2010-09-15,holo,SM5
3332,8SMC,C,Q5S007,TKL,LRRK2_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,290.31,81.86,1201,1327,NaN,ELECTRON MICROSCOPY,4.02,2024-01-03,holo,GDP;TVT
3333,6NYB,A,P15056,TKL,BRAF_HUMAN,Inactive,DFGin,BLBplus,Chelix-out,SaltBr-out,...,56.00,-87.07,805,-26,SEP365;SEP729,ELECTRON MICROSCOPY,4.10,2019-10-09,holo,AGS;ADP;LCJ
3334,9EDC,A,P17612,AGC,KAPCA_HUMAN,Active,DFGin,BLAminus,Chelix-in,SaltBr-in,...,273.36,51.23,350,1,SEP139;TPO197;SEP338,ELECTRON MICROSCOPY,4.18,2025-04-23,holo,ATP


In [21]:
filtered = filtered_df[['structure_id', 'chain_id', 'canonical_uniprot', 'canonical_gene_name', 
                                 'kinase_family', 'structural_label', 'activity_label',
                                    'spatial_label', 'dihedral_label', 'chelix_label',
                                    'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
                                    'asp_residue', 'phe_residue', 'gly_residue', 'lys_glu_distance', 'saltbridge_distance',
                                    'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
                                    'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
                                    'ligands', 'ligand_type', 'phosphorylations', 'method', 'resolution', 'deposition_date',
                                    ]
            ].sort_values(by = ["kinase_family", "canonical_gene_name", "deposition_date"]).reset_index(drop=True)

filtered.to_csv(f"wt{pho}/human_kinases_wt{pho}_filtered.csv", index=False)

In [22]:
pho = "nophos_nocov"

kinase_base = pd.read_csv(f"wt_{pho}/human_kinases_wt_{pho}.csv")
filtered_kinase_base = kinase_base.dropna(subset = ['structure_id', 'chain_id', 'canonical_uniprot', 'canonical_gene_name', 
                                 'kinase_family', 'activity_label',
                                    'spatial_label', 'dihedral_label', 'chelix_label',
                                    'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
                                    'asp_residue', 'phe_residue', 'gly_residue',
                                    'ligands', 'ligand_type', 'method', 'resolution', 'deposition_date'])

print(filtered_kinase_base.shape)

# uniprot_id_to_keep = list(set([k.split("_")[0] for k in canonical_sequences.keys()]))
# filtered_kinase_base = kinase_base[kinase_base['canonical_uniprot'].isin(uniprot_id_to_keep)]
filtered_kinase_base['structural_label'] = filtered_kinase_base['ligands'].apply(
    lambda x: 'apo' if x == 'No_ligand' else 'holo'
)

filtered_apo = filtered_kinase_base[filtered_kinase_base['structural_label'] == 'apo']

filtered_kinase_base['ligand_label'] = filtered_kinase_base['ligands'].apply(
    lambda x: ";".join([lig.split(":")[0] for lig in x.split(";")])
)

# further filter down based on labels, extract the higer resolution ones 

filtered_holo = (
    filtered_kinase_base[filtered_kinase_base['structural_label'] == 'holo']
    .sort_values(by='resolution')  # Sort by resolution ascending
    .drop_duplicates(
        subset=[
            'canonical_uniprot', 'kinase_family', 'activity_label', 'spatial_label',
            'dihedral_label', 'chelix_label', 'saltbr_label', 
            'actloop_nt_label', 'actloop_ct_label', 'spine_label',
            'asp_residue', 'phe_residue', 'gly_residue',
            'ligand_label', 'ligand_type'
        ],
        keep='first'  # Keep the first (lowest resolution) after sorting
    )
)

(2825, 40)


/tmp/ipykernel_1502285/3854189610.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_kinase_base['structural_label'] = filtered_kinase_base['ligands'].apply(
/tmp/ipykernel_1502285/3854189610.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_kinase_base['ligand_label'] = filtered_kinase_base['ligands'].apply(


In [23]:
print(filtered_apo.shape, filtered_holo.shape)
filtered_df = pd.concat([filtered_apo, filtered_holo], ignore_index=True)
filtered_df

(236, 41) (1989, 42)


,structure_id,chain_id,canonical_uniprot,kinase_family,canonical_gene_name,activity_label,spatial_label,dihedral_label,chelix_label,saltbr_label,...,phe_psi,phe_chi1,phe_chi2,sequence_length,first_residue,method,resolution,deposition_date,structural_label,ligand_label
0,1B6C,B,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,25.60,300.29,176.12,342,162,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
1,1B6C,D,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,25.60,300.53,176.04,342,162,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
2,1B6C,F,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,25.63,300.41,176.19,342,162,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
3,1B6C,H,P36897,TKL,TGFR1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,25.39,300.35,176.04,342,162,X-RAY DIFFRACTION,2.60,1999-06-15,apo,NaN
4,1BLX,A,Q00534,CMGC,CDK6_HUMAN,Inactive,DFGin,BLBplus,Chelix-out,SaltBr-out,...,168.28,58.33,85.97,326,1,X-RAY DIFFRACTION,1.90,1999-06-01,apo,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2220,8U7H,C,Q5S007,TKL,LRRK2_HUMAN,Active,DFGin,BLAminus,Chelix-in,SaltBr-in,...,68.13,318.52,-87.88,1201,1327,ELECTRON MICROSCOPY,3.80,2024-01-31,holo,GDP;A0T
2221,8DGT,B,Q02750,STE,MP2K1_HUMAN,Inactive,DFGin,BLBplus,Chelix-out,SaltBr-out,...,157.06,44.99,-83.68,414,-21,ELECTRON MICROSCOPY,3.90,2023-07-05,holo,AGS;LCJ;GNP
2222,3OMV,B,P04049,TKL,RAF1_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-in,...,35.62,285.33,59.46,307,312,X-RAY DIFFRACTION,4.00,2010-09-15,holo,SM5
2223,8SMC,C,Q5S007,TKL,LRRK2_HUMAN,Inactive,DFGin,BLAminus,Chelix-in,SaltBr-out,...,54.05,290.31,81.86,1201,1327,ELECTRON MICROSCOPY,4.02,2024-01-03,holo,GDP;TVT


In [24]:
filtered = filtered_df[['structure_id', 'chain_id', 'canonical_uniprot', 'canonical_gene_name', 
                                 'kinase_family', 'structural_label', 'activity_label',
                                    'spatial_label', 'dihedral_label', 'chelix_label',
                                    'saltbr_label', 'actloop_nt_label', 'actloop_ct_label', 'spine_label',
                                    'asp_residue', 'phe_residue', 'gly_residue', 'lys_glu_distance', 'saltbridge_distance',
                                    'spine_distance', 'dfg_hrd_distance', 'asp_phi', 'asp_psi', 'asp_chi1',
                                    'asp_chi2', 'phe_phi', 'phe_psi', 'phe_chi1', 'phe_chi2',
                                    'ligands', 'ligand_type', 'method', 'resolution', 'deposition_date',
                                    ]
            ].sort_values(by = ["kinase_family", "canonical_gene_name", "deposition_date"]).reset_index(drop=True)

filtered.to_csv(f"wt_{pho}/human_kinases_wt_{pho}_filtered.csv", index=False)

## Prepare Kinase structures in to Files

Now we have high-qaulity functionaly-relevant kinases left for building the benchmarks

In [25]:
kinase_base = pd.read_csv("wt_nophos_nocov/human_kinases_wt_nophos_nocov_filtered.csv")
kinase_base = kinase_base.dropna(subset=kinase_base.columns)
kinase_all = pd.read_csv("wt_nocov/human_kinases_wt_nocov_filtered.csv")
#kinase_all = kinase_all.dropna()

value_counts = kinase_all['canonical_uniprot'].value_counts()
repeated_values = value_counts[value_counts > 1].index.tolist()

kinase_base.shape, kinase_all.shape, len(repeated_values), len(value_counts)

((2225, 34), (3336, 35), 167, 217)

In [26]:
# calculate the number of unique labels in each column
unique_counts = kinase_all.nunique()
unique_counts

structure_id           2915
chain_id                 25
canonical_uniprot       217
canonical_gene_name     217
kinase_family             8
structural_label          2
activity_label            2
spatial_label             3
dihedral_label            8
chelix_label              2
saltbr_label              2
actloop_nt_label          2
actloop_ct_label          2
spine_label               2
asp_residue             187
phe_residue             194
gly_residue             190
lys_glu_distance        635
saltbridge_distance     751
spine_distance          410
dfg_hrd_distance        623
asp_phi                2358
asp_psi                2592
asp_chi1               2531
asp_chi2               2955
phe_phi                2416
phe_psi                2276
phe_chi1               2368
phe_chi2               2620
ligands                2747
ligand_type              22
phosphorylations        161
method                    2
resolution              520
deposition_date         879
dtype: int64

In [27]:
def summarize_activity(df, folder, write_ind_csv=False, write_gene_groups=False):
    """
    Groups the dataframe by `canonical_gene_name`, then for each group,
    counts how many are:
      - apo + active
      - apo + inactive
      - holo + active
      - holo + inactive
    
    Assumes:
      - 'activity_label' column has values like 'active', 'inactive'
      - 'ligands' column, where 'No_ligand' indicates apo
    """
    summary = []

    for gene, group in df.groupby("canonical_gene_name"):
        gene_family = group["kinase_family"].iloc[0]
        apo_active   = ((group["ligands"] == "No_ligand") & (group["activity_label"] == "Active")).sum()
        apo_inactive = ((group["ligands"] == "No_ligand") & (group["activity_label"] == "Inactive")).sum()
        holo_active  = ((group["ligands"] != "No_ligand") & (group["activity_label"] == "Active")).sum()
        holo_inactive= ((group["ligands"] != "No_ligand") & (group["activity_label"] == "Inactive")).sum()

        summary.append({
            "kinase_family": gene_family,
            "canonical_gene_name": gene,
            "apo_active": apo_active,
            "apo_inactive": apo_inactive,
            "holo_active": holo_active,
            "holo_inactive": holo_inactive,
        })

    summary_df = pd.DataFrame(summary)

    group1 = summary_df[(summary_df['apo_active'] >= 1) & (summary_df['apo_inactive'] >= 1)]['canonical_gene_name']
    group2 = summary_df[(summary_df['apo_active'] >= 1) & (summary_df['holo_inactive'] >= 1)]['canonical_gene_name']
    group3 = summary_df[(summary_df['apo_inactive'] >= 1) & (summary_df['holo_active'] >= 1)]['canonical_gene_name']
    group4 = summary_df[(summary_df['holo_active'] >= 1) & (summary_df['holo_inactive'] >= 1)]['canonical_gene_name']
    # come up with a group 5 that stores anything that have at least 10 structures and do not fall in any of the previous distributions
    group5_temp = summary_df[~summary_df['canonical_gene_name'].isin(group1) & ~summary_df['canonical_gene_name'].isin(group2) & ~summary_df['canonical_gene_name'].isin(group3) & ~summary_df['canonical_gene_name'].isin(group4)]
    group5 = group5_temp[group5_temp['apo_active'] + group5_temp['apo_inactive'] + group5_temp['holo_active'] + group5_temp['holo_inactive'] >= 10]['canonical_gene_name']

    if write_ind_csv:
        
        group1_df_apo_active = df[df['canonical_gene_name'].isin(group1) & (df['structural_label'] == "apo") & (df['activity_label'] == "Active")]
        group1_df_apo_inactive = df[df['canonical_gene_name'].isin(group1) & (df['structural_label'] == "apo") & (df['activity_label'] == "Inactive")]
        group1_df = pd.concat([group1_df_apo_active, group1_df_apo_inactive], ignore_index=True)
        group1_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        print(f"Group 1 size: {group1_df.shape}")
        print(f"Group 1 genes: {group1_df['canonical_gene_name'].nunique()}")
        group1_df.to_csv(f"{folder}/group1_apo_active_inactive.csv", index=False)
        
        group2_df_apo = df[df['canonical_gene_name'].isin(group2) & (df['structural_label'] == "apo") & (df['activity_label'] == "Active")]
        group2_df_holo = df[df['canonical_gene_name'].isin(group2) & (df['structural_label'] == "holo") & (df['activity_label'] == "Inactive")]
        group2_df = pd.concat([group2_df_apo, group2_df_holo], ignore_index=True)
        group2_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        print(f"Group 2 size: {group2_df.shape}")
        print(f"Group 2 genes: {group2_df['canonical_gene_name'].nunique()}")
        group2_df.to_csv(f"{folder}/group2_apo_active_holo_inactive.csv", index=False)
        
        group3_df_apo = df[df['canonical_gene_name'].isin(group3) & (df['structural_label'] == "apo") & (df['activity_label'] == "Inactive")]
        group3_df_holo = df[df['canonical_gene_name'].isin(group3) & (df['structural_label'] == "holo") & (df['activity_label'] == "Active")]
        group3_df = pd.concat([group3_df_apo, group3_df_holo], ignore_index=True)
        group3_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        print(f"Group 3 size: {group3_df.shape}")
        print(f"Group 3 genes: {group3_df['canonical_gene_name'].nunique()}")
        group3_df.to_csv(f"{folder}/group3_apo_inactive_holo_active.csv", index=False)
        
        group4_df_holo_active = df[df['canonical_gene_name'].isin(group4) & (df['structural_label'] == "holo") & (df['activity_label'] == "Active")]
        group4_df_holo_inactive = df[df['canonical_gene_name'].isin(group4) & (df['structural_label'] == "holo") & (df['activity_label'] == "Inactive")]
        group4_df = pd.concat([group4_df_holo_active, group4_df_holo_inactive], ignore_index=True)
        group4_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        print(f"Group 4 size: {group4_df.shape}")
        print(f"Group 4 genes: {group4_df['canonical_gene_name'].nunique()}")
        group4_df.to_csv(f"{folder}/group4_holo_active_inactive.csv", index=False)

        # Group 5: At least 2 distinct PDB IDs
        group5_df = df[df['canonical_gene_name'].isin(group5)]
        group5_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        print(f"Group 5 size: {group5_df.shape}")
        print(f"Group 5 genes: {group5_df['canonical_gene_name'].nunique()}")
        group5_df.to_csv(f"{folder}/group5_gene_with_one_activity_type.csv", index=False)


        # COMBINE ALL DF
        combined_df = pd.concat([group1_df, group2_df, group3_df, group4_df], ignore_index=True)
        combined_df.drop_duplicates(inplace=True)
        
        combined_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
        combined_df.to_csv(f"{folder}/combined_groups.csv", index=False)
        print(f"Combined groups size: {combined_df.shape}")

    if write_gene_groups:
        
        with open(f"{folder}/gene_groups.txt", "w") as f:
            f.write("Group 1 (Apo Active + Apo Inactive):\n")
            f.write("\n".join(group1) + "\n\n")
            f.write("Group 2 (Apo Active + Holo Inactive):\n")
            f.write("\n".join(group2) + "\n\n")
            f.write("Group 3 (Apo Inactive + Holo Active):\n")
            f.write("\n".join(group3) + "\n\n")
            f.write("Group 4 (Holo Active + Holo Inactive):\n")
            f.write("\n".join(group4) + "\n")
            f.write("Group 5 (One Activity Type):\n")
            f.write("\n".join(group5) + "\n")

    return pd.DataFrame(summary)

In [28]:
df_base = summarize_activity(kinase_base, "wt_nophos_nocov", write_ind_csv=True, write_gene_groups=True)
df_phos = summarize_activity(kinase_all, "wt_nocov", write_ind_csv=True, write_gene_groups=True)

Group 1 size: (108, 34)
Group 1 genes: 13
Group 2 size: (349, 34)
Group 2 genes: 20
Group 3 size: (389, 34)
Group 3 genes: 18
Group 4 size: (1282, 34)
Group 4 genes: 53
Group 5 size: (540, 34)
Group 5 genes: 21
Combined groups size: (1430, 34)
Group 1 size: (173, 35)
Group 1 genes: 22
Group 2 size: (570, 35)
Group 2 genes: 33
Group 3 size: (723, 35)
Group 3 genes: 34
Group 4 size: (2307, 35)
Group 4 genes: 77


/tmp/ipykernel_1502285/1818638824.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group5_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)
/tmp/ipykernel_1502285/1818638824.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group5_df.sort_values(by=["kinase_family", "canonical_gene_name", "deposition_date"], inplace=True)


Group 5 size: (447, 35)
Group 5 genes: 19
Combined groups size: (2615, 35)


## Extraction of Sequences

In [29]:
# Extract gene groups from the file
with open(f"wt_nocov/gene_groups.txt", "r") as f:
    phos_lines = f.readlines()

with open(f"wt_nophos_nocov/gene_groups.txt", "r") as f:
    nophos_lines = f.readlines()

lines = phos_lines + nophos_lines

gene_groups = []

for line in lines:
    if line.startswith("Group"): continue
    elif line.strip():  # Ignore empty lines
        gene_name = line.strip()
        gene_groups.append(gene_name)

gene_groups = list(set(gene_groups))
print(f"Total unique genes in groups: {len(gene_groups)}")

# read the FASTA file and extract sequences
from Bio import SeqIO
def extract_sequences(fasta_file, gene_groups):
    sequences = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        if record.id.split("|")[0] in gene_groups:
            sequences[record.id] = str(record.seq)
    return sequences

fasta_file = "../step_1_uniprot_extraction/dunbrack_active_kinases.fasta"
canonical_sequences = extract_sequences(fasta_file, gene_groups)

with open("kinase_pair_sequences.fasta", "w") as f:
    for k, v in canonical_sequences.items():
        f.write(f">{k}\n{v}\n")

Total unique genes in groups: 107
